# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [121]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "df.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "STATUS"

pd.set_option("display.max_columns", None)

In [122]:
df = pd.read_csv(RAW)
df.head()

,ID,MONTHS_BALANCE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,STATUS
0,5008804,0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
1,5008804,-1,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
2,5008804,-2,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
3,5008804,-3,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
4,5008804,-4,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0


## 1. Dados faltantes e Pré-tratamento de colunas

*Identificando colunas com valores nulos*

In [123]:
df.isnull().sum()

ID                          0
MONTHS_BALANCE              0
CODE_GENDER                 0
FLAG_OWN_CAR                0
FLAG_OWN_REALTY             0
CNT_CHILDREN                0
AMT_INCOME_TOTAL            0
NAME_INCOME_TYPE            0
NAME_EDUCATION_TYPE         0
NAME_FAMILY_STATUS          0
NAME_HOUSING_TYPE           0
DAYS_BIRTH                  0
DAYS_EMPLOYED               0
FLAG_WORK_PHONE             0
FLAG_PHONE                  0
FLAG_EMAIL                  0
OCCUPATION_TYPE        240048
CNT_FAM_MEMBERS             0
STATUS                      0
dtype: int64

**Interpretação**

Há 240048 nulos na coluna OCCUPATION_TYPE

---

*Verificando valores da coluna OCCUPATION_TYPE*

In [124]:
df['OCCUPATION_TYPE'].value_counts()

OCCUPATION_TYPE
Laborers                 131572
Core staff                77112
Sales staff               70362
Managers                  67738
Drivers                   47678
High skill tech staff     31768
Accountants               27223
Medicine staff            26691
Cooking staff             13416
Security staff            12400
Cleaning staff            11399
Private service staff      6714
Low-skill Laborers         3623
Secretaries                3149
Waiters/barmen staff       2557
HR staff                   1686
IT staff                   1319
Realty agents              1260
Name: count, dtype: int64

**Interpretação**

Nota-se que os valores não nulos se referem à cargos dos candidatos contratados.

Hipótese: valores nulos são de candidatos não contratados?
________________________________________________________________________________________________________________________________

*Tratando os nulos da coluna OCCUPATION_TYPE*

**Decisão:** 

O dicionário de dados informa que, se o número na coluna DAYS_EMPLOYED for positivo, significa que a pessoa está desempregada.

Para essa condição, os nulos da coluna OCCUPATION_TYPE serão substituídos por "Unemployed".

O restante dos nulos serão substituir por "Unknown", pois não se tem informações sobre a ocupação dessas pessoas.

In [125]:
for item in df[df.DAYS_EMPLOYED > 0].index:
  df.loc[item, "OCCUPATION_TYPE"] = "Unemployed"

df.fillna('Unknown', inplace=True)

df.isnull().sum()

ID                     0
MONTHS_BALANCE         0
CODE_GENDER            0
FLAG_OWN_CAR           0
FLAG_OWN_REALTY        0
CNT_CHILDREN           0
AMT_INCOME_TOTAL       0
NAME_INCOME_TYPE       0
NAME_EDUCATION_TYPE    0
NAME_FAMILY_STATUS     0
NAME_HOUSING_TYPE      0
DAYS_BIRTH             0
DAYS_EMPLOYED          0
FLAG_WORK_PHONE        0
FLAG_PHONE             0
FLAG_EMAIL             0
OCCUPATION_TYPE        0
CNT_FAM_MEMBERS        0
STATUS                 0
dtype: int64

Não há mais valores nulos

---

*Tratando o tipo de dado da coluna CNT_FAM_MEMBERS*

Alterando para int64

In [126]:
df.CNT_FAM_MEMBERS = df.CNT_FAM_MEMBERS.astype('int64')
df.CNT_FAM_MEMBERS.dtype

dtype('int64')

---

*Tratando a coluna DAYS_BIRTH*

Esta coluna mostra a idade em dias a partir da coleta dos dados.

In [127]:
# Passando o valor de dias para anos, e transformando em inteiro
df.DAYS_BIRTH = (df.DAYS_BIRTH / -365).astype('int64')

# Renomeando a coluna DAYS_BIRTH para AGE
df.rename(columns={'DAYS_BIRTH': 'AGE'}, inplace=True)

df.AGE.head()

0    32
1    32
2    32
3    32
4    32
Name: AGE, dtype: int64

---

*Tratando a coluna DAYS_EMPLOYED*

Pelo histograma da etapa 1.2, verifica-se que o range dos valores é bem grande a partir do zero.

In [128]:
# Verificando os valores únicos positivos

df.DAYS_EMPLOYED[df.DAYS_EMPLOYED > 0].unique()

array([365243])

**Interpretação**

Nota-se que há apenas 1 valor positivo (365243) para vários clientes (indicando que o cliente está desempregado), o que se trata de uma flag.

In [129]:
# Alterando os valores positivos da coluna DAYS_EMPLOYED para 0, ou seja, não está empregado

df.DAYS_EMPLOYED = df.DAYS_EMPLOYED.apply(lambda x: 0 if x > 0 else x)

# Passando o valor de dias para anos, e transformando em inteiro
df.DAYS_EMPLOYED = (df.DAYS_EMPLOYED / -365).astype('int64')

# Renomeando a coluna DAYS_EMPLOYED para YEARS_EMPLOYED
df.rename(columns={'DAYS_EMPLOYED': 'YEARS_EMPLOYED'}, inplace=True)

df.YEARS_EMPLOYED

0         12
1         12
2         12
3         12
4         12
          ..
777710     5
777711     5
777712     5
777713     5
777714     5
Name: YEARS_EMPLOYED, Length: 777715, dtype: int64

---

*Tratando a coluna AMT_INCOME_TOTAL*

Como a renda possui uma cauda longa à direita (muitos valores baixos/médios e poucos valores extremamente altos), a aplicação do logaritmo aproxima a distribuição de uma curva normal, reduz o impacto de outliers sem descartar informações e melhora o desempenho de algoritmos como Regressão Logística, Redes Neurais e Naive Bayes.

In [130]:
# Aplicação da transformação logarítmica

df.AMT_INCOME_TOTAL = np.log1p(df['AMT_INCOME_TOTAL'])

---

Criando a coluna de renda per capita

In [131]:
df['INCOME_PER_MEMBER'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

df.INCOME_PER_MEMBER.head()

0    6.482856
1    6.482856
2    6.482856
3    6.482856
4    6.482856
Name: INCOME_PER_MEMBER, dtype: float64

---

*Excluindo as colunas ID e MONTHS_BALANCE*

As colunas ID, MONTHS_BALANCE possuem valores muito diversos e podem atrapalhar em alguns modelos.

In [132]:
df.drop(columns=['ID', 'MONTHS_BALANCE'], inplace=True)
df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,AGE,YEARS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,STATUS,INCOME_PER_MEMBER
0,M,Y,Y,0,12.965712,Working,Higher education,Civil marriage,Rented apartment,32,12,1,0,0,Unknown,2,0,6.482856
1,M,Y,Y,0,12.965712,Working,Higher education,Civil marriage,Rented apartment,32,12,1,0,0,Unknown,2,0,6.482856
2,M,Y,Y,0,12.965712,Working,Higher education,Civil marriage,Rented apartment,32,12,1,0,0,Unknown,2,0,6.482856
3,M,Y,Y,0,12.965712,Working,Higher education,Civil marriage,Rented apartment,32,12,1,0,0,Unknown,2,0,6.482856
4,M,Y,Y,0,12.965712,Working,Higher education,Civil marriage,Rented apartment,32,12,1,0,0,Unknown,2,0,6.482856


---

*Verificando valores duplicados*

In [133]:
df.duplicated().sum()

np.int64(767887)

---

*Excluindo valores duplicados*

In [134]:
df.drop_duplicates(inplace=True)

df.reset_index(drop=True, inplace=True)

df.duplicated().sum()

np.int64(0)

In [135]:
df.shape

(9828, 18)

## 2. Definição da variável alvo

A coluna ALVO será a coluna STATUS

In [136]:
y = df[TARGET]

## 3. Normalização / padronização

**Escolha do Escalonador**


Para modelos baseados em árvores não há necessidade de escalonador, porém, para modelos baseados em distância é recomendado sua utilização.

Foi escolhido o **StandardScaler** para as colunas **numéricas**, pois essas colunas apresentam valores bem diferentes.

Aplicando o escalonador elas ficam todas com a mesma escala (com média para 0 e desvio padrão para 1).

---

*Aplicando o **StandardScaler** nas variáveis numéricas de interesse*

In [137]:
# Separando as colunas numéricas
numericas = ['AMT_INCOME_TOTAL', 'AGE', 'YEARS_EMPLOYED', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS']

# Instanciando o escalonador
scaler = StandardScaler()

# Escalonando
df_scaled = scaler.fit_transform(df[numericas])

# Transformando em dataframe
df_scaled = pd.DataFrame(df_scaled, columns=numericas)

# Excluindo as colunas numéricas do df original
df.drop(columns=df[numericas], inplace = True)

# Juntando os dataframes
df = pd.concat([df, df_scaled], axis = 1)
df.shape

df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,STATUS,INCOME_PER_MEMBER,AMT_INCOME_TOTAL,AGE,YEARS_EMPLOYED,CNT_CHILDREN,CNT_FAM_MEMBERS
0,M,Y,Y,Working,Higher education,Civil marriage,Rented apartment,1,0,0,Unknown,0,6.482856,2.006494,-0.956283,1.059145,-0.558271,-0.201770
1,M,Y,Y,Working,Secondary / secondary special,Married,House / apartment,0,0,0,Security staff,0,5.815359,-0.749305,1.302080,-0.372449,-0.558271,-0.201770
2,F,N,Y,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,0,12.506181,1.057895,0.780919,0.422881,-0.558271,-1.266753
3,F,N,Y,Pensioner,Higher education,Separated,House / apartment,0,0,0,Unemployed,0,12.554971,1.158611,1.562660,-0.849647,-0.558271,-1.266753
4,M,Y,Y,Working,Higher education,Married,House / apartment,1,1,1,Accountants,0,6.253090,1.057895,0.259758,-0.531515,-0.558271,-0.201770


In [138]:
df.shape

(9828, 18)

## 4. Feature engineering

Há colunas que podem ser binarizadas e há colunas categóricas podem ser tratadas para ficarem com o mesmo padrão das outras colunas.

_____

*Binarizando as colunas FLAG_OWN_CAR, FLAG_OWN_REALTY*

Conforme visto na anáise exploratória de dados, as colunas FLAG_OWN_CAR, FLAG_OWN_REALTY possuem valores dicotômicos (N e Y) que podem ser binarizados (0 e 1)

Essa binarização pode ser feita pelo Label Encoder, porém, neste caso será feito de forma manual.

In [139]:
df.FLAG_OWN_CAR = df.FLAG_OWN_CAR.apply(lambda x: 0 if x=='N' else 1)
df.FLAG_OWN_REALTY = df.FLAG_OWN_REALTY.apply(lambda x: 0 if x=='N' else 1)

----

*Aplicando o **One-Hot Encoder** nas colunas categóricas*

In [140]:
# Separando as colunas categóricas
categoricas = []

for i in df.columns:
  if df[i].dtype == 'str':
    categoricas.append(i)

print(f'Colunas categóricas: {categoricas}')

# Aplicando o hot encoder
hot = []

for i in df.columns:
  hot = pd.get_dummies(df[categoricas], prefix = 'hot')


# Mesclando os  dataframes
df = pd.concat([df, hot], axis=1)

# Excluindo as colunas categóricas originais
df.drop(columns=categoricas, inplace=True)

df.head()

Colunas categóricas: ['CODE_GENDER', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


,FLAG_OWN_CAR,FLAG_OWN_REALTY,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,STATUS,INCOME_PER_MEMBER,AMT_INCOME_TOTAL,AGE,YEARS_EMPLOYED,CNT_CHILDREN,CNT_FAM_MEMBERS,hot_F,hot_M,hot_Commercial associate,hot_Pensioner,hot_State servant,hot_Student,hot_Working,hot_Academic degree,hot_Higher education,hot_Incomplete higher,hot_Lower secondary,hot_Secondary / secondary special,hot_Civil marriage,hot_Married,hot_Separated,hot_Single / not married,hot_Widow,hot_Co-op apartment,hot_House / apartment,hot_Municipal apartment,hot_Office apartment,hot_Rented apartment,hot_With parents,hot_Accountants,hot_Cleaning staff,hot_Cooking staff,hot_Core staff,hot_Drivers,hot_HR staff,hot_High skill tech staff,hot_IT staff,hot_Laborers,hot_Low-skill Laborers,hot_Managers,hot_Medicine staff,hot_Private service staff,hot_Realty agents,hot_Sales staff,hot_Secretaries,hot_Security staff,hot_Unemployed,hot_Unknown,hot_Waiters/barmen staff
0,1,1,1,0,0,0,6.482856,2.006494,-0.956283,1.059145,-0.558271,-0.201770,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,1,1,0,0,0,0,5.815359,-0.749305,1.302080,-0.372449,-0.558271,-0.201770,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
2,0,1,0,1,1,0,12.506181,1.057895,0.780919,0.422881,-0.558271,-1.266753,True,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False
3,0,1,0,0,0,0,12.554971,1.158611,1.562660,-0.849647,-0.558271,-1.266753,True,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
4,1,1,1,1,1,0,6.253090,1.057895,0.259758,-0.531515,-0.558271,-0.201770,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [141]:
df.shape

(9828, 55)

## 5. Salvar dataset tratado

In [142]:
dataset_tratado = df.copy

PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED / "dataset_tratado.csv", index=False)